# HOMEWORK WEEK 6

#### In this week, a regression model will be built to create a regression model for predicting the car fuel efficiency (`fuel_efficiency_mpg`).


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv")

In [3]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


#### Data preparation

>###### Fill missing values with `0.0`.
>###### Use `DictVectorizer(sparse=True)` to turn the dataframes into matrices.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   model_year           10000 non-null  int64  
 1   origin               10000 non-null  str    
 2   fuel_type            10000 non-null  str    
 3   drivetrain           10000 non-null  str    
 4   num_doors            10000 non-null  int64  
 5   engine_displacement  10000 non-null  int64  
 6   num_cylinders        10000 non-null  int64  
 7   horsepower           9123 non-null   float64
 8   vehicle_weight       10000 non-null  int64  
 9   acceleration         9736 non-null   float64
 10  fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(3), int64(5), str(3)
memory usage: 859.5 KB


In [5]:
df.isna().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [6]:
df.describe()

,model_year,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
count,10000.000000,10000.00000,10000.000000,10000.000000,9123.000000,10000.000000,9736.000000,10000.000000
mean,1999.549500,3.50400,2268.807000,6.035200,254.446016,4286.754000,18.888044,29.997700
std,14.207925,0.95209,183.447355,0.542578,22.844613,266.212491,1.279543,2.930245
min,1975.000000,2.00000,1590.000000,4.000000,171.000000,3320.000000,14.000000,19.800000
25%,1987.000000,3.00000,2150.000000,6.000000,239.000000,4110.000000,18.000000,28.000000
50%,2000.000000,4.00000,2270.000000,6.000000,254.000000,4280.000000,18.900000,30.000000
75%,2012.000000,4.00000,2390.000000,6.000000,270.000000,4470.000000,19.700000,31.900000
max,2024.000000,5.00000,3110.000000,8.000000,334.000000,5460.000000,23.800000,41.200000


In [7]:
# Fill missing value with 0
for c in ['horsepower', 'acceleration']:
    df[c] = df[c].fillna(0)

In [8]:
df.isna().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [9]:
# Split the data to train/val/test by 60%/20%/20%
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [10]:
# Reset index
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [11]:
y_train = df_train['fuel_efficiency_mpg'].to_numpy()
y_val = df_val['fuel_efficiency_mpg'].to_numpy()
y_test = df_test['fuel_efficiency_mpg'].to_numpy()

In [12]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

## Question 1

##### Train a decision tree regressor to predict the `fuel_efficiency_mpg`.

> ##### Train a model with `max_depth=1`.

#### Which ***original feature*** is used for splitting the data?

In [13]:
# Convert the train data into a dictionary
train_dicts = df_train.to_dict(orient='records')

# Change the dataframes into matrices
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)

In [14]:
# Observe all the features used
dv.get_feature_names_out()

array(['acceleration', 'drivetrain=All-wheel drive',
       'drivetrain=Front-wheel drive', 'drivetrain=Rear-wheel drive',
       'engine_displacement', 'fuel_type=Diesel', 'fuel_type=Gasoline',
       'fuel_type=Hybrid', 'horsepower', 'model_year', 'num_cylinders',
       'num_doors', 'origin=Asia', 'origin=Europe', 'origin=USA',
       'vehicle_weight'], dtype=object)

In [15]:
# Train a Decision Tree Regression model
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with 

In [16]:
# Transform the validation dataset
val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [17]:
# Predict with the decision tree model
y_train_pred = dt.predict(X_train)
y_val_pred = dt.predict(X_val)

# Calculate the rmse on the datasets
rmse_train = root_mean_squared_error(y_train, y_train_pred)
rmse_val = root_mean_squared_error(y_val, y_val_pred)

print(f'RMSE for training  : {rmse_train:.4f}')
print(f'RMSE for validation: {rmse_val:.4f}')


RMSE for training  : 2.6406
RMSE for validation: 2.6726


In [18]:
# Look inside the tree
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



#### `model_year` with threshold `1997.50` is used by the model to split the data.

## Question 2

##### Train a random forest regressor model with these parameters:
>##### `n_estimators=10`
>##### `random_state=1`
>##### `n_jobs=-1` (optional - to make training faster by using all available cores)

#### What's the `RMSE` of this model on the validation data?

In [19]:
# Configure the random forest regressor model
rf_regr = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf_regr.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

In [20]:
# Train and transform the datasets
train_dicts = df_train.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

# Predict by using random forest regressor model
y_pred = rf_regr.predict(X_val)

rmse = root_mean_squared_error(y_val, y_pred)
print(f'The RMSE of the model on the validation is {rmse:.3f}')

The RMSE of the model on the validation is 1.833


##### However, there is a slight difference with the options given (`1.837`). This is possibly due to missing one parameter in the given question: `max_depth=20`.

In [21]:
# Add one more criteria in the training model
rf_regr = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_regr.fit(X_train, y_train)

train_dicts = df_train.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

y_pred = rf_regr.predict(X_val)

rmse = root_mean_squared_error(y_val, y_pred)
print(f'The RMSE of the model on the validation is {rmse:.3f}')

The RMSE of the model on the validation is 1.837


## Question 3

##### Train a random forest regressor model with different `n_estimators` parameter:
>##### `n_estimators=[10, 50, 100, 150]`
>##### `random_state=1`
>##### `n_jobs=-1` (optional)

#### Which value gives the ***lowest validation RMSE***?

In [22]:
# Train and transform the datasets again to avoid state pollution
train_dicts = df_train.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [23]:
# Experiment with different n_estimators
n_estimators = [10, 50, 100, 150]

rmses = []

for n in n_estimators:
    rf_regr = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf_regr.fit(X_train, y_train)

    y_pred = rf_regr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)

    rmses.append((n, round(rmse, 3)))

rmses_sorted = sorted(rmses, key=lambda x:x[1])
rmses_sorted

[(150, 1.769), (100, 1.77), (50, 1.774), (10, 1.833)]

In [24]:
print(f'The lowest validation RMSE is {rmses_sorted[0][1]} when n_estimator = {rmses_sorted[0][0]}.')

The lowest validation RMSE is 1.769 when n_estimator = 150.


## Question 4

##### Train a random forest regressor model to find best `max_depth` parameter:
>##### Try different values of `max_depth=[10, 15, 20, 25]`
>##### Also try different values of `n_estimators=[10, 50, 100, 150]`
>##### `random_state=1`
>##### `n_jobs=-1` (optional)

#### What's the ***best `max_depth`***, using the mean `RMSE`?

In [25]:
# Experiment with different max_depth and n_estimators
max_depths = [10, 15, 20, 25]
n_estimators = [10, 50, 100, 150]

rmses = []

for depth in max_depths:
    for n in n_estimators:
        rf_regr = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1)
        rf_regr.fit(X_train, y_train)
    
        y_pred = rf_regr.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
    
        rmses.append((n, depth, rmse))

# Sort the list in place
rmses.sort(key=lambda x:x[2])
rmses

[(150, 10, 1.7437162901130996),
 (100, 10, 1.7439861944908972),
 (50, 10, 1.747519928302535),
 (150, 15, 1.7667522783622136),
 (100, 15, 1.7678723617281948),
 (150, 25, 1.7679271415004252),
 (100, 25, 1.7683198864738403),
 (150, 20, 1.768476341618912),
 (100, 20, 1.7689177484873317),
 (50, 15, 1.770883049718161),
 (50, 25, 1.7721232000066),
 (50, 20, 1.7723767866064453),
 (10, 10, 1.7823475646138716),
 (10, 15, 1.8313896883458138),
 (10, 25, 1.8368517467554855),
 (10, 20, 1.8373138746453748)]

In [26]:
print(f'The lowest validation RMSE is {rmses[0][2]:.4f} when n_estimator = {rmses[0][0]} and max_depth = {rmses[0][1]}.')

The lowest validation RMSE is 1.7437 when n_estimator = 150 and max_depth = 10.


## Question 5

##### Find the most important feature by training the model with these parameter:
>##### `n_estimators=10`
>##### `max_depth=20`
>##### `random_state=1`
>##### `n_jobs=-1` (optional)

#### What's the ***most important feature***? 
##### (among `vehicle_weight`, `horsepower`, `acceleration`, `engine_displacement`)

In [27]:
# Get the trained feature names
feature_names = dv.get_feature_names_out()

# Get the importance scores from the model
importances = rf_regr.feature_importances_

# Display in a sorted dataframe
df_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False).reset_index(drop=True)

df_importance.head(10)

,feature,importance
0,model_year,0.294907
1,vehicle_weight,0.202990
2,fuel_type=Gasoline,0.156226
3,horsepower,0.076011
4,engine_displacement,0.063908
5,acceleration,0.056335
6,origin=USA,0.041768
7,drivetrain=Front-wheel drive,0.020176
8,num_doors,0.019399
9,fuel_type=Hybrid,0.018935


In [28]:
target_features = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']

df_target = df_importance[df_importance['feature'].isin(target_features)]

feature, score = df_target.iloc[0]

display(df_target)
print(f'The most important feature is "{feature}" with importance score of {score:.4f}.')

,feature,importance
1,vehicle_weight,0.202990
3,horsepower,0.076011
4,engine_displacement,0.063908
5,acceleration,0.056335


The most important feature is "vehicle_weight" with importance score of 0.2030.


## Question 6

##### Train a XGBoost model by:
>##### 1. Install XGBoost.
>##### 2. Create `DMatrix` for train and validation datasets.
>##### 3. Create a `watchlist`.
>##### 4. Train a model with given parameters for 100 rounds.
>##### 5. Change `eta` from `0.3` to `0.1`.

#### Which `eta` leads to the ***best `RMSE`*** score on the validation dataset?

In [29]:
%pip install xgboost 


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
import xgboost as xgb

In [31]:
# Train and transform the datasets again to avoid state pollution
train_dicts = df_train.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [32]:
# Get the trained feature names
features = list(dv.get_feature_names_out())

# Create DMatrix for train and validation datasets.
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

In [33]:
# Create a watchlist
watchlist = [(dtrain, 'train'), (dval, 'val')]

In [34]:
%%capture output

# Given parameter
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',

    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

evals_result = dict()

# Train the xgb model
model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                  verbose_eval=5,
                  evals=watchlist,
                  evals_result=evals_result)

In [35]:
# Print the captured output
output.show()

[0]	train-rmse:2.40653	val-rmse:2.46782
[5]	train-rmse:1.59328	val-rmse:1.76216
[10]	train-rmse:1.47125	val-rmse:1.73205
[15]	train-rmse:1.41908	val-rmse:1.73512
[20]	train-rmse:1.39368	val-rmse:1.73766
[25]	train-rmse:1.35837	val-rmse:1.74359
[30]	train-rmse:1.33243	val-rmse:1.74548
[35]	train-rmse:1.29290	val-rmse:1.75198
[40]	train-rmse:1.25326	val-rmse:1.76296
[45]	train-rmse:1.21527	val-rmse:1.77244
[50]	train-rmse:1.18645	val-rmse:1.77517
[55]	train-rmse:1.14797	val-rmse:1.78668
[60]	train-rmse:1.10265	val-rmse:1.79401
[65]	train-rmse:1.06590	val-rmse:1.80143
[70]	train-rmse:1.04604	val-rmse:1.80444
[75]	train-rmse:1.02727	val-rmse:1.80682
[80]	train-rmse:0.99487	val-rmse:1.81308
[85]	train-rmse:0.96970	val-rmse:1.81600
[90]	train-rmse:0.94986	val-rmse:1.82135
[95]	train-rmse:0.91805	val-rmse:1.82544
[99]	train-rmse:0.90106	val-rmse:1.82646


In [36]:
%%capture output

# Train a new model with eta = 0.1
xgb_params_new = {
    'eta': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',

    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

evals_result_new = dict()

model_new = xgb.train(xgb_params_new, dtrain, num_boost_round=100,
                  verbose_eval=5,
                  evals=watchlist,
                  evals_result=evals_result_new)

In [37]:
# Print the captured output
output.show()

[0]	train-rmse:2.74739	val-rmse:2.78635
[5]	train-rmse:2.12109	val-rmse:2.20428
[10]	train-rmse:1.80592	val-rmse:1.93152
[15]	train-rmse:1.64286	val-rmse:1.80346
[20]	train-rmse:1.55847	val-rmse:1.75076
[25]	train-rmse:1.50918	val-rmse:1.72751
[30]	train-rmse:1.47544	val-rmse:1.71667
[35]	train-rmse:1.44978	val-rmse:1.71287
[40]	train-rmse:1.43074	val-rmse:1.71289
[45]	train-rmse:1.41543	val-rmse:1.71310
[50]	train-rmse:1.40368	val-rmse:1.71358
[55]	train-rmse:1.39009	val-rmse:1.71570
[60]	train-rmse:1.37884	val-rmse:1.71688
[65]	train-rmse:1.37172	val-rmse:1.71812
[70]	train-rmse:1.36283	val-rmse:1.71919
[75]	train-rmse:1.35099	val-rmse:1.72057
[80]	train-rmse:1.34578	val-rmse:1.72100
[85]	train-rmse:1.33480	val-rmse:1.72192
[90]	train-rmse:1.32546	val-rmse:1.72303
[95]	train-rmse:1.32292	val-rmse:1.72324
[99]	train-rmse:1.31192	val-rmse:1.72481


In [38]:
# Compare the results for both eta=0.3 and eta=0.1
etas = [0.3, 0.1]
results = [evals_result, evals_result_new]

for eta, result in zip(etas, results):
    
    # Determine the best rmse score and iteration number
    val_rmse = result['val']['rmse']
    
    best_rmse = np.min(val_rmse)
    best_iteration = np.argmin(val_rmse) + 1

    # Determine the final trained result
    val_rmse_final = result['val']['rmse'][-1]

    print(f'When eta = {eta},')
    print(f"Best Validation RMSE: {best_rmse:.4f} at round {best_iteration}.")
    print(f"Final Validation RMSE: {val_rmse_final:.4f} after training.\n")


When eta = 0.3,
Best Validation RMSE: 1.7320 at round 11.
Final Validation RMSE: 1.8265 after training.

When eta = 0.1,
Best Validation RMSE: 1.7122 at round 40.
Final Validation RMSE: 1.7248 after training.

